## Data Engineering Module 4 - Medallion Architecture with Delta Lake

![med_new](./Assets/med_new.png)

### Loading CSV files into the Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading the CSV files into a Dataframe

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')
display(df.limit(100))

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8
SO49171,1,2021-01-01,Mariah Foster,mariah21@adventure-works.com,"Road-250 Black, 48",1,2181.5625,174.525
SO49172,1,2021-01-01,Brian Howard,brian23@adventure-works.com,"Road-250 Red, 44",1,2443.35,195.468
SO49173,1,2021-01-01,Linda Alvarez,linda19@adventure-works.com,"Mountain-200 Silver, 38",1,2071.4196,165.7136
SO49174,1,2021-01-01,Gina Hernandez,gina4@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4196,165.7136
SO49178,1,2021-01-01,Beth Ruiz,beth4@adventure-works.com,"Road-550-W Yellow, 44",1,1000.4375,80.035
SO49179,1,2021-01-01,Evan Ward,evan13@adventure-works.com,"Road-550-W Yellow, 38",1,1000.4375,80.035
SO49175,1,2021-01-01,Margaret Guo,margaret24@adventure-works.com,"Road-250 Red, 52",1,2443.35,195.468
SO49180,1,2021-01-01,Mitchell Yuan,mitchell6@adventure-works.com,"Road-650 Black, 58",1,782.99,62.6392
SO49176,1,2021-01-01,Shawn Sharma,shawn11@adventure-works.com,"Mountain-200 Silver, 46",1,2071.4196,165.7136
SO49177,1,2021-01-01,Barbara Chande,barbara44@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4196,165.7136


### Defining the Schema for the Bronze Layer Dataframe

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronzeSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

### Creating the Bronze Layer Dataframe

In [0]:
bronze_df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv', schema=bronzeSchema)
display(bronze_df.limit(100))

SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,Email,Item,Quantity,UnitPrice,Tax
SO49171,1,2021-01-01,Mariah Foster,mariah21@adventure-works.com,"Road-250 Black, 48",1,2181.5625,174.525
SO49172,1,2021-01-01,Brian Howard,brian23@adventure-works.com,"Road-250 Red, 44",1,2443.35,195.468
SO49173,1,2021-01-01,Linda Alvarez,linda19@adventure-works.com,"Mountain-200 Silver, 38",1,2071.4197,165.7136
SO49174,1,2021-01-01,Gina Hernandez,gina4@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4197,165.7136
SO49178,1,2021-01-01,Beth Ruiz,beth4@adventure-works.com,"Road-550-W Yellow, 44",1,1000.4375,80.035
SO49179,1,2021-01-01,Evan Ward,evan13@adventure-works.com,"Road-550-W Yellow, 38",1,1000.4375,80.035
SO49175,1,2021-01-01,Margaret Guo,margaret24@adventure-works.com,"Road-250 Red, 52",1,2443.35,195.468
SO49180,1,2021-01-01,Mitchell Yuan,mitchell6@adventure-works.com,"Road-650 Black, 58",1,782.99,62.6392
SO49176,1,2021-01-01,Shawn Sharma,shawn11@adventure-works.com,"Mountain-200 Silver, 46",1,2071.4197,165.7136
SO49177,1,2021-01-01,Barbara Chande,barbara44@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4197,165.7136


### Creating the Bronze Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Bronze;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5744918647270700>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Bronze;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driv

### Storing the Bronze Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
bronzeDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/bronze_sales_orders'
bronze_df.write.format('delta').mode('overwrite').save(bronzeDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
bronze_df.write.format('delta').saveAsTable('Bronze.bronze_sales_orders')

### Creating the Silver Layer by Cleaning and Transforming the Data

### Creating the Silver Layer Dataframe

In [0]:
silver_df = spark.read.format("delta").table("Bronze.bronze_sales_orders")
silver_df.show(10)

### Cleaning the Silver Layer Dataframe

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.dropDuplicates()
silver_df = silver_df.withColumn('Tax', col('UnitPrice') * 0.08)
silver_df = silver_df.withColumn('Tax', col('Tax').cast("float"))

### Creating the Silver Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Silver;

### Storing the Silver Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
silverDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/silver_sales_orders'
silver_df.write.format('delta').mode('overwrite').save(silverDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
silver_df.write.format('delta').saveAsTable('Silver.silver_sales_orders')

### Creating the Gold Layer by Aggregating the Data

### Creating the Gold Layer Dataframe

In [0]:
gold_df = spark.read.format("delta").table("Silver.silver_sales_orders")
gold_df.show(10)

### Aggregating Yearly Sales Data

In [0]:
yearlySales = gold_df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

### Creating the Gold Schema in Unity Catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Gold;

### Storing the Gold Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
goldDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/gold_sales_orders'
yearlySales.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
yearlySales.write.format('delta').saveAsTable('Gold.gold_sales_orders')